<a href="https://colab.research.google.com/github/nat-coutoune/BioPython_XMeeting/blob/main/day3_bioblast/Day_3_BioBlast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![Logo](../../../resources/logo_horizontal_xm26.jpeg)

# **Day 3: BioBlast**
---
There are two ways to query Blast using BioPython: one via the internet and the other by making the comparison locally. Now let's take a closer look at how each of them works.



**Quick Review of the BLAST Concept:**

What is BLAST? A fundamental algorithm for finding regions of local similarity between biological sequences.
Why use BLAST? It is crucial for functional annotation of genes and proteins, identification of homologs (genes or proteins with a common ancestor), and for preliminary evolutionary studies.
Key Parameters:

* E-value: Represents the expected number of alignments with a score equal to or better than that found by chance in a database of the same size. The smaller the E-value (closer to zero), the more statistically significant the alignment.

* Score: The sum of close matches and mismatches in the alignment, penalizing gaps. Higher scores indicate greater similarity.

* Identity: A percentage of residues (nucleotides or amino acids) that are identical between homologous sequences.

* Alignment: The visual representation of matches, mismatches, and gaps between sequences.

## 1. Running BLAST over the Internet
---

**BLAST Online: Why and When to Use It?**

**Advantages**: Does not require software installation or local databases. Access to the most up-to-date public databases from NCBI. Convenient for one-off or small-volume queries.     
**Disadvantages:** Depends on internet connection. Can be slow for many queries. NCBI has request limits that must be respected to avoid temporary blocks. For large volumes, local BLAST is preferable.


We use the function `blast` in the `Bio.Blast` module to call the online version of BLAST.

The NCBI guidelines state:

  *  Do not contact the server more often than once every 10 seconds.
  * Do not poll for any single RID more often than once a minute.
  *  Use the URL parameter email and tool, so that the NCBI can contact you if there is a problem.
  *  Run scripts weekends or between 9 pm and 5 am Eastern time on weekdays if more than 50 searches will be submitted.

`Blast.qblast follows the first two points automatically. To fulfill the third point, set the Blast.email variable (the Blast.tool variable is already set to "biopython" by default):

In [ ]:
# Installing BioPython
!pip install biopython

In [ ]:
# Importing Blast
from Bio import Blast
Blast.email = "your@mail.com"

The `qblast` function has three non-optional arguments:

 *   The first argument is **a lower case string**.  Currently qblast only works with **blastn, blastp, blastx, tblast and tblastx**.
 *   The second argument specifies the **databases** to search against. Again, the options for this are available on NCBI’s BLAST Help pages.
 *   The third argument is a **string containing your query sequence**:  can be the sequence itself, the sequence in fasta format, or an identifier like a GI number.


The qblast function also takes a number of other option arguments, which are basically analogous to the different parameters you can set on the BLAST web page. We’ll just highlight a few of them here:

  *  The argument url_base sets the base URL for running BLAST over the internet. By default it connects to the NCBI.
  *  The qblast function can return the BLAST results in various formats, which you can choose with the optional format_type keyword: "XML", "HTML", "Text", "XML2", "JSON2", or "Tabular".
  *  The argument expect sets the expectation or e-value threshold.

  For more about the optional BLAST arguments, we refer you to the NCBI’s own documentation, or that built into Biopython:


In [ ]:
help(Blast.qblast)

Basic usage:

In [ ]:
result_blast = Blast.qblast("blastn", "nt", "8332116")

In [ ]:
type(result_blast)

http.client.HTTPResponse

Alternatively, if we have our query sequence already in a FASTA formatted file, we just need to open the file and read in this record as a string, and use that as the query argument.

But first, let's learn how to upload sequences from Google Drive:

In [ ]:
# To access to Drive files
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Changing the path
%cd /content/drive/My Drive/projetos/mini-curso_biopython/
!pwd
!ls

And finally open the fasta sequence:

In [ ]:
fasta_string = open("S288C_YJR070C_LIA1_genomic.fsa").read()
result_blast_nt = Blast.qblast("blastn", "nt", fasta_string)

We could also have read in the FASTA file as a SeqRecord and then supplied just the sequence itself:

In [ ]:
from Bio import SeqIO
record = SeqIO.read("S288C_YJR070C_LIA1_genomic.fsa", "fasta")
result_blast = Blast.qblast("blastn", "nt", record.seq)

Supplying just the sequence means that BLAST will assign an identifier for your sequence automatically. You might prefer to call format on the SeqRecord object to make a FASTA string (which will include the existing identifier):

In [ ]:
from Bio import Blast
from Bio import SeqIO
records = SeqIO.parse("lia1_Cmetapsilosis.gbk", "genbank")
record = next(records)
result_blast = Blast.qblast("blastn", "nt", format(record, "fasta"))

## Saving BLAST results   



The `qblast()` function gives you BLAST results as a byte stream, typically XML. We strongly recommend saving this stream to a local file before parsing it. This not only makes debugging easier by avoiding repeated, slow online searches but also helps conserve NCBI's computing resources. Keep in mind that a stream can only be read once; subsequent attempts will yield an empty result.


In [ ]:
with open("my_blast.xml", "wb") as out_stream:
    out_stream.write(result_blast.read())
result_blast.close()

After doing this, the results are in the file my_blast.xml and result_blast has had all its data extracted (so we closed it). However, the parse function of the BLAST parser takes a file-like object, so we can just open the saved file for input as bytes:

In [ ]:
result_blast = open("my_blast.xml", "rb")

## Parsing BLAST output

As mentioned above, BLAST can generate output in various formats, such as XML, HTML, and plain text. Originally, Biopython had parsers for BLAST plain text and HTML output, as these were the only output formats offered at the time. These parsers have now been removed from Biopython, as the BLAST output in these formats kept changing, each time breaking the Biopython parsers. Nowadays, Biopython can parse BLAST output in the XML format, the XML2 format, and tabular format.

You can get BLAST output in XML format in various ways. For the parser, it doesn’t matter how the output was generated, as long as it is in the XML format.

   * You can use Biopython to run BLAST over the internet
   * You can use Biopython to run BLAST locally
   * You can do the BLAST search yourself on the NCBI site through your web browser, and then save the results. You need to choose XML as the format in which to receive the results, and save the final BLAST page you get (you know, the one with all of the interesting results!) to a file.
   * You can also run BLAST locally without using Biopython, and save the output in a file. Again, you need to choose XML as the format in which to receive the results.

The important point is that you do not have to use Biopython scripts to fetch the data in order to be able to parse it. Doing things in one of these ways, you then need to get a file-like object to the results. In Python, a file-like object or handle is just a nice general way of describing input to any info source so that the info can be retrieved using read() and readline() functions.


In [ ]:
from Bio import Blast
result_blast = Blast.qblast("blastn", "nt", "8332116")

If instead you ran BLAST some other way, and have the BLAST output (in XML format) in the file my_blast.xml, all you need to do is to open the file for reading (as bytes):

In [ ]:
result_blast = open("my_blast.xml", "rb")

Now that we’ve got a data stream, we are ready to parse the output. The code to parse it is really quite small. If you expect a single BLAST result (i.e., you used a single query):

Just like Bio.SeqIO and Bio.Align, we have a pair of input functions, read and parse, where read is for when you have exactly one object, and parse is an iterator for when you can have lots of objects – but instead of getting SeqRecord or Alignment objects, we get BLAST record objects.

To be able to handle the situation where the BLAST file may be huge, containing thousands of results, Blast.parse() returns an iterator. In plain English, an iterator allows you to step through the BLAST output, retrieving BLAST records using a `for` loop:

In [ ]:
from Bio import Blast
for blast_record in blast_records:
  pass  # Do something with blast_record

Note though that you can step through the BLAST records only once. Usually, from each BLAST record you would save the information that you are interested in.   
Alternatively, you can use blast_records as a list, for example by extracting one record by index, or by calling len or print on blast_records. The parser will then automatically iterate over the records and store them:

In [ ]:
from Bio import Blast
blast_records = Blast.parse("my_blast.xml")
len(blast_records)  # this causes the parser to iterate over all records
blast_records[0].query.description

'XM_067691878.1 Candida metapsilosis LIA1 (I9W82_002965), partial mRNA'

If your BLAST file is huge though, you may run into memory problems trying to save them all in a list.

If you start iterating over the records before using blast_records as a list, the parser will first reset the file stream to the beginning of the data to ensure that all records are neing read. In those cases, you can explicitly read the records into a list by calling blast_records = blast_records[:] before iterating over them. After reading in the records, it is safe to iterate over them or use them as a list.

Instead of opening the file yourself, you can just provide the file name:



In [ ]:
from Bio import Blast
with Blast.parse("my_blast.xml") as blast_records:
     for blast_record in blast_records:
         pass  # Do something with blast_record

In this case, Biopython opens the file for you, and closes it as soon as the file is not needed any more (while it is possible to simply use blast_records = Blast.parse("my_blast.xml"), it has the disadvantage that the file may stay open longer than strictly necessary, thereby wasting resources).

You can print the records to get a quick overview of their contents:

In [ ]:
from Bio import Blast
with Blast.parse("my_blast.xml") as blast_records:
     print(blast_records)

Usually, you’ll be running one BLAST search at a time. Then, all you need to do is to pick up the first (and only) BLAST record in blast_records:

In [ ]:
from Bio import Blast
blast_records = Blast.parse("my_blast.xml")
blast_record = next(blast_records)

## The BLAST Records, Record, and Hit classes


### The BLAST Records class

A single BLAST output file can contain output from multiple BLAST queries. In Biopython, the information in a BLAST output file is stored in an Bio.Blast.Records object. This is an iterator returning one Bio.Blast.Record object for each query. The Bio.Blast.Records object has the following attributes describing the BLAST run:

  *  source: The input data from which the Bio.Blast.Records object was constructed (this could be a file name or path, or a file-like object).
  *  program: The specific BLAST program that was used (e.g., ’blastn’).
  * version: The version of the BLAST program (e.g., ’BLASTN 2.2.27+’).
  *  reference: The literature reference to the BLAST publication.
  *  db: The BLAST database against which the query was run (e.g., ’nr’).
  * query: A SeqRecord object which may contain some or all of the following information:
      *  query.id: SeqId of the query;
      *  query.description: Definition line of the query;
      *  query.seq: The query sequence.
  *   param: A dictionary with the parameters used for the BLAST run. You may find the following keys in this dictionary:
     *   'matrix': the scoring matrix used in the BLAST run (e.g., ’BLOSUM62’) (string);
     *   'expect': threshold on the expected number of chance matches (float);
     *   'include': e-value threshold for inclusion in multipass model in psiblast (float);
     *   'sc-match': score for matching nucleotides (integer);
     *   'sc-mismatch': score for mismatched nucleotides (integer;
     *   'gap-open': gap opening cost (integer);
     *   'gap-extend': gap extension cost (integer);
     *   'filter': filtering options applied in the BLAST run (string);
     *   'pattern': PHI-BLAST pattern (string);
     *   'entrez-query': Limit of request to Entrez query (string).
  *   mbstat: A dictionary with Mega BLAST search statistics.

For our example, we find:

In [ ]:
blast_records

In [ ]:
blast_records.source

In [ ]:
blast_records.program

In [ ]:
blast_records.version

In [ ]:
blast_records.reference

In [ ]:
blast_records.db

In [ ]:
blast_records.param

### The BLAST Record class

A Bio.Blast.Record object stores the information provided by BLAST for a single query. The Bio.Blast.Record class inherits from list, and is essentially a list of Bio.Blast.Hit objects. A Bio.Blast.Record object has the following two attributes:

 *   query: A SeqRecord object which may contain some or all of the following information:
        query.id: SeqId of the query;
        query.description: Definition line of the query;
        query.seq: The query sequence.
 *   stat: A dictionary with statistical data of the BLAST hit. You may find the following keys in this dictionary:
        'db-num': number of sequences in BLAST db (integer);
        'db-len': length of BLAST db (integer);
        'hsp-len': effective HSP (High Scoring Pair) length (integer);
        'eff-space': effective search space (float);
        'kappa': Karlin-Altschul parameter K (float);
        'lambda': Karlin-Altschul parameter Lambda (float);
        'entropy': Karlin-Altschul parameter H (float)
 *   message: Some (error?) information.

Continuing with our example:

In [ ]:
blast_record

<Bio.Blast.Record query.id='Query_2023377'; 50 hits>

In [ ]:
blast_record.query

SeqRecord(seq=Seq(None, length=954), id='Query_2023377', name='<unknown name>', description='XM_067691878.1 Candida metapsilosis LIA1 (I9W82_002965), partial mRNA', dbxrefs=[])

In [ ]:
blast_record.stat

In [ ]:
print(blast_record)

As the Bio.Blast.Record class inherits from list, you can use it as such. For example, you can iterate over the record:

In [ ]:
for hit in blast_record:
  hit

To check how many hits the blast_record has, you can simply invoke Python’s len function:

In [ ]:
len(blast_record)

50

Like Python lists, you can retrieve hits from a Bio.Blast.Record using indices:

In [ ]:
print(blast_record[0])  # retrieves the top hit
print(blast_record[-1])  # retrieves the last hit

To retrieve multiple hits from a Bio.Blast.Record, you can use the slice notation. This will return a new Bio.Blast.Record object containing only the sliced hits:

In [ ]:
blast_slice = blast_record[:3]  # slices the first three hits
print(blast_slice)

### The BLAST Hit class
Each Bio.Blast.Hit object in the blast_record list represents one BLAST hit of the query against a target.

In [ ]:
hit = blast_record[0]
hit.target

We can get a summary of the hit by printing it:

In [ ]:
 print(blast_record[3])

You see that we’ve got the essentials covered here:

*    A hit is always for one query; the query ID and description are shown at the top of the summary.
*    A hit consists of one or more alignments of the query against one target sequence. The target information is shown next is the summary. As shown above, the target can be accessed via the target attribute of the hit.
*    Finally, there’s a table containing quick information about the alignments each hit contains. In BLAST parlance, these alignments are called "High-scoring Segment Pairs", or HSPs. Each row in the table summarizes one HSP, including the HSP index, e-value, bit score, span (the alignment length including gaps), query coordinates, and target coordinates.

The Bio.Blast.Hit class is a subclass of Bio.Align.Alignments, and therefore in essence is a list of Bio.Align.Alignment objects. In particular when aligning nucleotide sequences against the genome, the Bio.Blast.Hit object may consist of more than one Bio.Align.Alignment if a particular query aligns to more than one region of a chromosome. For protein alignments, usually a hit consists of only one alignment, especially for alignments of highly homologous sequences.

In [ ]:
type(hit)
from Bio.Align import Alignments
isinstance(hit, Alignments)
len(hit)

1

In Biopython, the difference between a "record" (singular) and "records" (plural) largely depends on the context of the module you are using and what it's designed to return. However, there's a very common pattern, especially with Bio.SeqIO and Bio.Blast.NCBIXML:

### **A "Record" (Singular): Represents a Single Biological Entry.**
This is typically an object that holds all the information for one sequence, one alignment, one BLAST result, one phylogenetic tree, etc.  
It's like a single row in a spreadsheet, containing various fields (attributes) for that specific entry.   
Examples:
* Bio.SeqIO.read() returns a single SeqRecord object. This object contains the sequence (.seq), its ID (.id), description (.description), features (.features), and annotations (.annotations).
* Bio.Blast.NCBIXML.read() returns a single BlastRecord object. This object represents all the hits for a single query sequence in a BLAST XML output.
* Bio.Align.Alignment (from Bio.Align.parse()): Each object represents a single alignment record.
Bio.Phylo.Tree (from Bio.Phylo.read()): Represents a single phylogenetic tree.

### **"Records" (Plural): Represents an Iterable of Multiple Biological Entries.**

This usually refers to an iterator or a list that contains multiple "record" objects.     
It's designed for situations where your input file or query might yield many individual entries. Iterators are memory-efficient because they process items one by one, rather than loading everything into memory at once.  

Examples:

* Bio.SeqIO.parse() returns an iterator of SeqRecord objects. You use a for loop to go through each SeqRecord in a multi-FASTA, multi-GenBank, etc., file.

* Bio.Blast.NCBIXML.parse() returns an iterator of BlastRecord objects. This is used when a single BLAST XML output file contains results for multiple query sequences (e.g., if you submitted a multi-FASTA query to BLAST).
* Bio.Align.parse() returns an iterator of Bio.Align.Alignment objects.
            Bio.Phylo.parse() returns an iterator of Bio.Phylo.Tree objects.

**Analogy:**

Imagine you have a box of delicious chocolates:
* A "Record" (singular) is like one single chocolate from the box. You can examine its type, filling, shape, etc.
* "Records" (plural), as returned by a parse() function, is like the entire box of chocolates where you can pick them out one by one. You don't necessarily take all of them out at once; you just take one, enjoy it, then take the next.

#### **Key Takeaway and Practical Implication:**

The main difference lies in whether the function is expecting/returning one single entry or potentially many entries.  

Use read() when you expect your input source (file, handle) to contain exactly one record. If it finds more than one, it will raise an error.
Use parse() when your input source might contain multiple records. It will return an iterator, allowing you to process them efficiently, one by one, typically in a for loop.

This distinction is fundamental for efficient and robust scripting in Biopython, especially when dealing with large biological datasets.

### Sorting and filtering BLAST output
If the ordering of hits in the BLAST output file doesn’t suit your taste, you can use the sort method to resort the hits in the Bio.Blast.Record object. As an example, here we sort the hits based on the sequence length of each target, setting the reverse flag to True so that we sort in descending order.

In [ ]:
for hit in blast_record[:5]:
    print(f"{hit.target.id} {len(hit.target)}")

gi|2797497025|ref|XM_067691878.1| 954
gi|448523123|ref|XM_003868811.1| 954
gi|2325072772|ref|XM_051751086.1| 2208
gi|1919716256|ref|XM_036811576.1| 954
gi|2325475169|ref|XM_051817559.1| 954


This will sort blast_record in place. Use original_blast_record = blast_record[:] before sorting if you want to retain a copy of the original, unsorted BLAST output.

To filter BLAST hits based on their properties, you can use Python's built-in filter with the approciate callback function to evaluate each hit. The callback function must accept as its argument a single Hit object and return True or False. Here is an example in which we filter out Hit objects that only have one HSP:

## 2. Running Blast locally
---

### Introduction

Running BLAST locally has at least major two advantages:

 *  Local BLAST may be faster than BLAST over the internet;
 *  Local BLAST allows you to make your own database to search for sequences against.

Dealing with proprietary or unpublished sequence data can be another reason to run BLAST locally. You may not be allowed to redistribute the sequences, so submitting them to the NCBI as a BLAST query would not be an option.

Unfortunately, there are some major drawbacks too – installing all the bits and getting it setup right takes some effort:

  * Local BLAST requires command line tools to be installed.
  * Local BLAST requires (large) BLAST databases to be setup (and potentially kept up to date).


```
# Importa o módulo subprocess para executar comandos de linha de comando
import subprocess
import os

# 1. Criar um arquivo FASTA para ser nosso pequeno banco de dados local de exemplo
db_fasta_content = """
>seq_ref_001 OrganismoA hypothetical gene
ATGCGTACGTACGTATGCGTACGTACGTACGT
>seq_ref_002 OrganismoB conserved region
ATGCATGCATGCATGCATGCATGCATGCATGC
>seq_ref_003 OrganismoC random sequence
TACGTACGTACGTACGTAAAACGTACGTACGT
"""
db_fasta_file = "local_db_sequences.fasta"
with open(db_fasta_file, "w") as f:
    f.write(db_fasta_content)
print(f"Arquivo '{db_fasta_file}' criado para o banco de dados local.")

# 2. Nome do banco de dados que será criado (apenas o prefixo)
local_db_name = "my_custom_nucl_db"

# 3. Comando makeblastdb para criar o banco de dados
# O tipo do banco de dados (-dbtype) é nucl (nucleotídeo) ou prot (proteína)
# -out define o prefixo dos arquivos do banco de dados
makeblastdb_command = ["makeblastdb", "-in", db_fasta_file, "-dbtype", "nucl", "-out", local_db_name]

print(f"\nExecutando makeblastdb para criar '{local_db_name}'...")
try:
    # Usamos subprocess.run para executar o comando e capturar a saída
    # check=True fará com que uma exceção seja levantada se o comando falhar
    result = subprocess.run(makeblastdb_command, capture_output=True, text=True, check=True)
    print("makeblastdb Saída Padrão:\n", result.stdout)
    if result.stderr:
        print("makeblastdb Saída de Erro:\n", result.stderr)
    print(f"Banco de dados '{local_db_name}' criado com sucesso!")
except subprocess.CalledProcessError as e:
    print(f"\nERRO ao criar o banco de dados BLAST local. Verifique a instalação do BLAST+ e os caminhos.")
    print(f"Erro: {e}")
    print(f"Saída de Erro do makeblastdb:\n{e.stderr}")
    # Limpeza em caso de erro
    if os.path.exists(db_fasta_file):
        os.remove(db_fasta_file)
    exit()

```

## Referências & futuras leituras
* [Página oficial do BioPython (Início)](https://biopython.org/docs/latest/index.html)
* [Página oficial do BioPython (BioBlast)](https://biopython.org/docs/latest/Tutorial/chapter_blast.html)
* [Post no Medium do Frederico Schmitt Kremer](https://medium.com/omixdata/introdu%C3%A7%C3%A3o-ao-biopython-parte-ii-utilizando-o-blast-com-o-python-64fad3a3be3b)



